<!-- nav -->
[← 9 · Constraints as data](09_Constraints_As_Data.ipynb) · [Home](../../README.md) · [11 · Constraints over collections →](11_Constraints_Over_Collections.ipynb)

# 10 · Answers you can trust

**The problem.** Loggers drop out: a battery dies, a radio loses signal, and a shipment arrives with no temperature on
file. What should "is it in range?" answer then? A TypeScript function compares `undefined` with `2.0` and says
`false`: the shipment is quietly rejected, or, with the test written the other way round, quietly accepted. A Python
one raises `TypeError`. Neither is an answer you can act on.

Simplifying an expression with the data bound answers with a literal, its value, when the data decides; when it
doesn't, what is left is the question still open, the residual, and the answer is **unknown**. This case study shows
how unknowns combine, how you decide what they mean, and why `1n` and `1.0` aren't equal.

In [1]:
import { Simplifiers, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import { cold_chain, show } from "./toolkit.js";

const store = cold_chain();
const Expression = store.Expressions.Framework.Expression;
const shipments: any[] = store.transaction(() => [
  store.Shipment((s: any) => s.id("S1").celsius(5.5).hours(30n)),
  store.Shipment((s: any) => s.id("S2").celsius(9.1).hours(80n)),
  store.Shipment((s: any) => s.id("S3").hours(20n))]); // its logger never reported
const s1 = shipments[0];
const shipment = E.variable("shipment");
const celsius = shipment.celsius;
const between = E.literal(2.0).le(celsius).and_(celsius.le(8.0)); // a writer, which other writers build on
const in_range = store.transaction(() => Expression((e: any) => e.root(between)));
const table = (constraint: any) =>
  Object.fromEntries(shipments.map((s: any) => [s.id.get(), Simplifiers.OfAny(constraint, { shipment: s }).value]));

<a id="T10.1"></a>
## 10.1 · Absent is unknown

In mbse-schemas, a property is either present or absent; there's no `undefined` value to compare. Reading an absent
property gives **unknown**, no literal, so the subterm stays in the residual and the simplification's value is unknown,
which TypeScript writes `null`, and a comparison with unknown is unknown:

In [2]:
console.log(table(in_range));
const s3 = shipments[2];
const left = store.transaction(() => Expression((e: any) => e.root(Simplifiers.OfAny(in_range, { shipment: s3 }).residual)));
console.log(show(left)); // what is left for S3: the question nobody has answered

{ S1: true, S2: false, S3: null }


S3's answer isn't false: it's that nobody knows, and what is left for S3 is the question itself, its residual. `has`
asks whether a property is present, which always has an answer:

In [3]:
const reported = store.transaction(() => Expression((e: any) => e.root(shipment.has("celsius"))));
console.log(show(reported));
console.log(table(reported));

has(shipment, 'celsius')


{ S1: true, S2: true, S3: false }


<a id="T10.2"></a>
## 10.2 · How unknowns combine

`and_`, `or_`, `not_` and `implies` follow **Kleene's three-valued logic**: an unknown operand doesn't matter when the
other one decides. `false and unknown` is false, whatever the unknown turns out to be; `true or unknown` is true.

In [4]:
const [late_or_out, ok_or_silent] = store.transaction(() => [
  Expression((e: any) => e.root(shipment.hours.gt(72n).or_(between.not_()))),
  Expression((e: any) => e.root(shipment.has("celsius").not_().or_(between)))]);
for (const constraint of [late_or_out, ok_or_silent]) {
  console.log(show(constraint));
  console.log(table(constraint));
}

or(gt(get(shipment, 'hours'), 72), not(and(le(2.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), 8.0))))


{ S1: false, S2: true, S3: null }


or(not(has(shipment, 'celsius')), and(le(2.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), 8.0)))


{ S1: true, S2: false, S3: true }


For S3, the first constraint is unknown: it was on time, so everything rests on a temperature nobody has. The second
decides S3 at its first operand, and doesn't evaluate the second.

`implies` reads "if ... then": a constraint about only some shipments holds trivially for the others:

In [5]:
const long_trips_cold = store.transaction(() => Expression((e: any) => e.root(shipment.hours.gt(72n).implies(shipment.celsius.le(8.0)))));
console.log(show(long_trips_cold));
console.log(table(long_trips_cold));

implies(gt(get(shipment, 'hours'), 72), le(get(shipment, 'celsius'), 8.0))


{ S1: true, S2: false, S3: true }


<a id="T10.3"></a>
## 10.3 · You decide what unknown means

Simplification never guesses: unknown is a residual, not a literal. The program asking decides, once, at the point where it acts. A dashboard that flags
shipments to inspect flags what isn't known to be fine; an audit that certifies shipments certifies only what is known
to be fine:

In [6]:
const inspect = shipments.filter((s: any) => Simplifiers.OfAny(in_range, { shipment: s }).value !== true).map((s: any) => s.id.get());
const certify = shipments.filter((s: any) => Simplifiers.OfAny(in_range, { shipment: s }).value === true).map((s: any) => s.id.get());
console.log(inspect, certify);

[ "S2", "S3" ] [ "S1" ]


Writing `if (Simplifiers.OfAny(...).value)` would treat unknown as false, which is the silent mistake this case study
started with. Compare with `=== true` or `=== false`.

<a id="T10.4"></a>
## 10.4 · No coercion

An integer and a float are different kinds of value, as they are in mbse-schemas, of different domains, as
[tutorial 2](02_One_Value_A_Domain_In_Each_Language.ipynb) shows: `1n` and `1.0` aren't equal, they're incomparable, so comparing them is unknown. That catches a classic slip. `celsius` is a float, so a constraint that
writes the limits as integers answers unknown for every shipment, rather than an answer that depends on the language:

In [7]:
const sloppy = store.transaction(() => Expression((e: any) => e.root(E.literal(2n).le(shipment.celsius).and_(shipment.celsius.le(8n)))));
console.log(table(sloppy));

{ S1: null, S2: null, S3: null }


JavaScript would have compared `2n <= 5.5` happily, and Python compares `2 <= 5.5` too, each by its own rules. A
constraint meant to give the same answer everywhere can't depend on either. Arithmetic is stricter still: adding an
integer and a float is an error, raised when the constraint is simplified:

In [8]:
const mixed = store.transaction(() => Expression((e: any) => e.root(shipment.hours.add(0.5).gt(72.0))));
try {
  Simplifiers.OfAny(mixed, { shipment: s1 }).value;
} catch (error) {
  console.log(String(error));
}

TypeError: add expects numbers of one domain, got int and float


<a id="T10.5"></a>
## 10.5 · What stays, and what raises

Unknown is for data that doesn't decide. A variable nothing binds is not a mistake but a question for later: it stays
in the residual, among the names the simplification still needs (case study 12). An operation Patterns doesn't define
stays too, named, so that you can ask why it wasn't simplified. An operand of the wrong kind is a mistake in the
constraint, and raises. `validate` (case study 9) reports the first two before the constraint is simplified.

In [9]:
const [divided, numbered] = store.transaction(() => [
  Expression((e: any) => e.root(E.operation("div", 1n, 2n))), Expression((e: any) => e.root(shipment.id.add(1n)))]);
console.log([...Simplifiers.OfAny(in_range, {}).free].sort()); // a question for later: nothing binds shipment
console.log(Simplifiers.OfAny(divided, {}).residues.map((residue) => residue.reason)); // no rule: it stays, named
try {
  Simplifiers.OfAny(numbered, { shipment: s1 });
} catch (error) { // a mistake in the constraint
  console.log(String(error));
}

KeyError: variable 'shipment' is not bound


NotImplementedError: 'div' is not an operation Patterns defines


TypeError: add expects numbers of one domain, got str and int


## Why it works this way

- **Unknown is not false.** Treating it as false fails good records silently, and treating it as true passes bad ones,
  so the program asking decides what it means
  ([PRI-007: three-valued checking on request](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-007)).
- **Kleene, because it's the logic of not knowing yet.** `false and unknown` is false because no value of the unknown
  could make it true: more data can't overturn an answer an expression gives.
- **No coercion, because languages disagree.** TypeScript, Python, MATLAB, Excel and C each convert numbers by their own rules,
  and a constraint must mean the same everywhere.

How expressions simplify, three-valued, in full:
[CAP-106: expressions evaluated three-valued](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-106.md#evaluation-in-mbse-expressions),
and why simplification rather than evaluation:
[PRI-046: simplification, not evaluation](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-046).

<!-- nav -->
[← 9 · Constraints as data](09_Constraints_As_Data.ipynb) · [Home](../../README.md) · [11 · Constraints over collections →](11_Constraints_Over_Collections.ipynb)